# Imports

In [ ]:
import pandas as pd
import requests

# Functions

In [ ]:
def load_uniprot_fasta(identifier): #loads fasta file for a given UniProt identifier
    link = "http://www.uniprot.org/uniprot/" + identifier + ".fasta"

    str_data = requests.get(link).content.decode('utf-8')
    fasta = str_data.split('>')
    fasta_all=[]
    for seq in fasta[1:]:
      temp = seq.splitlines()[1:]
      temp = ''.join(temp)
      fasta_all.append(temp)
    return fasta_all[0] # only canonical sequence


def substitute(uniprot, sequence, start, end):
        '''
        Input: Uniprots ID,  for 19 remaining AA substitutions from start to end

         Output: uniprot id	WT_sequence	mut_sequence	AA_orig	position	AA_targ	mutant
                  test_id	    AKIP	    CKIP	        A	      1       	C     	A1C
                  test_id	    AKIP	    DKIP	        A	      1	        D	      A1D
                  ...
        '''
        sequence_part = list(sequence[start-1:end-1]) # keep posit_range of sequence - example: 193-280

        AA_list = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
        mut_sequence, AA_targ, AA_orig, position, mutant = [], [], [], [], []

        for i, AA in enumerate(sequence_part):
                mut_seq = list(sequence)
                remaining_AA = AA_list.copy()
                remaining_AA.remove(AA)
                for k in remaining_AA:
                        mut_seq[start+i-1] = k
                        mut_sequence.append(''.join(mut_seq))
                        AA_orig.append(AA)
                        AA_targ.append(k)
                        position.append(start+i)
                        mutant.append( AA+str(start+i)+k)
        d = {'uniprot id': [uniprot]*len(AA_targ), 'WT_sequence' : sequence, 'mut_sequence':mut_sequence, 'AA_orig': AA_orig, 'position' : position, 'AA_targ' : AA_targ, 'mutant': mutant}
        df = pd.DataFrame(data =d)
        return df


def subst_download(uniprot, start=1, end=1):
        '''
        Input: Uniprots ID,  for 19 other AA substitutions from start to end
        '''
        # Download sequence from uniprot
        sequence = load_uniprot_fasta(uniprot)
        if end ==1:
            end= len(sequence) + 1
        df= substitute(uniprot, sequence, start, end)

        return df

# Substitute all positions with all alternative Amino Acids

## Download from Uniprot API

In [ ]:
uniprot_list = [ 'Q96GD4']
start = 1
end = 1 # 1 when I mutate the entire sequence
for uniprot in uniprot_list:
  df=subst_download(uniprot, start, end)
  df.to_csv(uniprot+ '_all.csv', index=False)

## Or input sequence

In [ ]:
start = 1
sequence = 'MPRYTVHVRGEWLAVPCQDAQLTV....' # fill in with own sequence
uniprot = 'example_uniprot'
end= len(sequence) + 1 # change if section of sequence only needed
df = substitute(uniprot, sequence, start, end)
df.to_csv(uniprot+ '_all.csv')